# Практические задачи: очистка и подготовка данных (Решения)
Задачи к лекции 5 «Очистка и подготовка данных» (`5.da.fillna.rus.pdf`): пропущенные значения, `dropna` и `fillna`, дубликаты, типы данных, выбросы, очистка строк и проверка согласованности.

Сложность: ⭐ — базовая, ⭐⭐ — средняя, ⭐⭐⭐ — повышенная.

В этом ноутбуке приведены решения. Сначала попробуйте решить задачи самостоятельно в `5.da.fillna.tasks.ipynb`.

## 0. Данные для практики

Во всех заданиях (если не сказано иное) используется таблица заказов интернет-магазина `df_raw`.

| Столбец | Значение |
| :--- | :--- |
| `order_id` | Номер заказа (должен быть уникальным) |
| `mijoz` | Имя клиента |
| `telefon` | Телефон клиента |
| `shahar` | Город доставки |
| `kategoriya` | Категория товара |
| `narx` | Цена за единицу, сум |
| `soni` | Количество |
| `summa` | Сумма заказа (`narx × soni`) |
| `buyurtma_sana` | Дата заказа |
| `yetkazish_sana` | Дата доставки |

Таблица специально «испорчена»: в ней есть почти все проблемы, разобранные на лекции.

Запустите ячейку ниже. **Каждое задание начинайте с `df = df_raw.copy()`**, если в нём не сказано работать с результатом предыдущего задания.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1004, 1005, 1006, 1007,
                 1008, 1009, 1009, 1010, 1011, 1012, 1013],
    'mijoz': ['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов', 'Камола Рахимова',
              'Камола Рахимова', None, 'Жасур Тошев', 'Нодира Исмоилова',
              'Шахзод Ахмедов', 'Малика Саидова', 'Малика Саидова', 'алишер каримов',
              'Сардор Нуриев', 'Зарина Холова', 'Отабек Юлдашев'],
    'telefon': ['+998 90 123-45-67', '998911234567', '90 555 44 33', '93-111-22-33',
                '93-111-22-33', '+998(94)7778899', '', '97 000 11 22',
                '12-34', '+998 99 888 77 66', '+998 99 888 77 66', '901234567',
                '+998 91 222 33 44', '95-333-44-55', '+998901112233'],
    'shahar': ['Ташкент', 'ташкент ', 'Самарканд', 'Бухара', 'Бухара', 'Tashkent',
               'г. Самарканд', 'N/A', 'Бухара', 'Самарканд', 'Самарканд', 'ТАШКЕНТ',
               'Наманган', '-', 'Наманган'],
    'kategoriya': ['Продукты', 'Электроника', 'Продукты', 'Одежда', 'Одежда', 'Продукты',
                   'Электроника', 'Одежда', 'Продукты', 'Одежда', 'Одежда', 'Электроника',
                   'Продукты', 'Одежда', 'Электроника'],
    'narx': ['12 000', '450000', '8000', '95000', '95000', '?', '520 000 сум', '120000',
             '1500000', '75000', '75000', '380000', '6 500', '110000', ''],
    'soni': [2, 1, np.nan, 1, 1, 3, 1, -2, 1, 2, 2, 1, 4, np.nan, 2],
    'summa': [24000, 450000, np.nan, 95000, 95000, np.nan, 520000, -240000,
              1500000, 150000, 150000, 400000, 26000, np.nan, np.nan],
    'buyurtma_sana': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-02',
                      '2026-09-03', '2026-09-03', '2026-09-04', '2026-09-04', '2026-09-05',
                      '2026-09-05', '2026-09-06', '2026-09-06', 'нет даты', '2026-09-07'],
    'yetkazish_sana': ['2026-09-02', '2026-09-04', '2026-09-03', '2026-09-05', '2026-09-05',
                       '2026-09-04', '2026-09-02', '2026-09-06', '2026-09-05', None,
                       '2026-09-07', '2026-09-08', '2026-09-07', None, '2026-09-09'],
})

df = df_raw.copy()   # работаем с копией, df_raw не изменяем
df

## 1. Поиск пропущенных значений

**Задача 1.1** ⭐ Выведите информацию о таблице с помощью `info()`. В каких столбцах есть пропуски по мнению pandas?

**Задача 1.2** ⭐ Посчитайте количество пропусков в каждом столбце и долю пропусков в процентах (округлите до 1 знака). Выведите результат одной таблицей с двумя столбцами: `kol_vo` и `dolya_%`.

**Задача 1.3** ⭐ Выведите все строки, в которых есть хотя бы один пропуск. Сколько их?

**Задача 1.4** ⭐⭐ Найдите **скрытые пропуски**. Выведите `value_counts(dropna=False)` для столбцов `shahar` и `narx`. Какие значения на самом деле означают «нет данных»?

**Задача 1.5** ⭐⭐ Замените все скрытые пропуски (`'N/A'`, `'-'`, `'?'`, `''`) на `np.nan`. Сравните количество пропусков **до** и **после** замены в одной таблице.

In [ ]:
# Задача 1.1
df = df_raw.copy()
df.info()
# По мнению pandas пропуски есть в: mijoz, soni, summa, yetkazish_sana

# Задача 1.2
report = pd.DataFrame({
    'kol_vo': df.isna().sum(),
    'dolya_%': (df.isna().mean() * 100).round(1),
})
print('\n', report)

# Задача 1.3
mask = df.isna().any(axis=1)
print('\nСтрок с пропусками:', mask.sum())
display(df[mask])

# Задача 1.4
print(df['shahar'].value_counts(dropna=False))
print(df['narx'].value_counts(dropna=False))
# Скрытые пропуски: 'N/A' и '-' в shahar, '?' и '' (пустая строка) в narx,
# а также пустая строка '' в telefon и 'нет даты' в buyurtma_sana

# Задача 1.5
before = df.isna().sum()
df = df.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
after = df.isna().sum()
print(pd.DataFrame({'до': before, 'после': after}))

## 2. Стратегии работы с пропусками: dropna и fillna

Перед заданиями выполните замену скрытых пропусков из задачи 1.5:
```python
df = df_raw.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
```

**Задача 2.1** ⭐ Сколько строк останется после каждого вызова? Выведите `shape` для:
* `df.dropna()`
* `df.dropna(subset=['soni'])`
* `df.dropna(how='all')`
* `df.dropna(thresh=9)`

Какой вариант потерял больше всего данных и почему?

**Задача 2.2** ⭐ Заполните пропуски в столбце `mijoz` значением `'Неизвестно'`, а в столбце `shahar` — **модой** (самым частым городом).

**Задача 2.3** ⭐⭐ Для столбца `soni` посчитайте среднее и медиану. Заполните пропуски медианой. Затем сравните среднее значение `soni` в трёх вариантах: пропуски удалены, пропуски заполнены нулём, пропуски заполнены медианой. Почему заполнение нулём — плохая идея для количества?

**Задача 2.4** ⭐⭐ Перед заполнением `soni` создайте столбец-флаг `soni_nan`, который показывает, было ли значение пропущено. Выведите строки, где флаг равен `True`.

**Задача 2.5** ⭐⭐ Дан ряд температур за неделю:
```python
temp = pd.Series([18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
                 index=pd.date_range('2026-09-01', periods=7))
```
Заполните пропуски тремя способами: `ffill()`, `bfill()` и `interpolate()`. Выведите результаты рядом в одной таблице. Почему последнее значение после `bfill()` осталось пустым?

**Задача 2.6** ⭐⭐⭐ Дана таблица товаров:
```python
tovar = pd.DataFrame({
    'kategoriya': ['Продукты', 'Продукты', 'Продукты', 'Одежда', 'Одежда', 'Одежда', 'Электроника', 'Электроника'],
    'narx': [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})
```
Заполните пропущенные цены **медианой своей категории** (через `groupby(...).transform('median')`). Сравните с заполнением общей медианой. Какой вариант разумнее?

In [ ]:
df = df_raw.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})

# Задача 2.1
print('Исходно:              ', df.shape)
print("dropna():             ", df.dropna().shape)
print("dropna(subset=soni):  ", df.dropna(subset=['soni']).shape)
print("dropna(how='all'):    ", df.dropna(how='all').shape)
print("dropna(thresh=9):     ", df.dropna(thresh=9).shape)
# Больше всего потерял dropna() без параметров: он удаляет строку,
# если пусто ХОТЯ БЫ одно значение, а пропуски разбросаны по разным строкам.

# Задача 2.2
df['mijoz'] = df['mijoz'].fillna('Неизвестно')
moda = df['shahar'].mode()[0]
print('\nМода города:', moda)
df['shahar'] = df['shahar'].fillna(moda)

# Задача 2.4 (флаг создаём ДО заполнения)
df['soni_nan'] = df['soni'].isna()
display(df[df['soni_nan']])

# Задача 2.3
print('Среднее soni:', df['soni'].mean(), '| медиана soni:', df['soni'].median())
print('Среднее, пропуски удалены:   ', df['soni'].dropna().mean().round(3))
print('Среднее, пропуски = 0:       ', df['soni'].fillna(0).mean().round(3))
print('Среднее, пропуски = медиана: ', df['soni'].fillna(df['soni'].median()).mean().round(3))
df['soni'] = df['soni'].fillna(df['soni'].median())
# Ноль означает «купили 0 штук», а не «не знаем». Он занижает среднее и сумму.

# Задача 2.5
temp = pd.Series([18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
                 index=pd.date_range('2026-09-01', periods=7))
print(pd.DataFrame({'исходно': temp, 'ffill': temp.ffill(),
                    'bfill': temp.bfill(), 'interpolate': temp.interpolate()}))
# После bfill последнее значение пустое: после него нет ни одного известного
# значения, которое можно было бы взять «из будущего».

# Задача 2.6
tovar = pd.DataFrame({
    'kategoriya': ['Продукты', 'Продукты', 'Продукты', 'Одежда', 'Одежда', 'Одежда', 'Электроника', 'Электроника'],
    'narx': [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})
tovar['po_gruppe'] = tovar['narx'].fillna(tovar.groupby('kategoriya')['narx'].transform('median'))
tovar['obshchaya'] = tovar['narx'].fillna(tovar['narx'].median())
print(tovar)
# Заполнение по группе разумнее: общая медиана (80 000) даёт хлеб за 80 000
# и телефон за 80 000. Медиана категории учитывает, что товары разные.

## 3. Дублирующиеся строки

**Задача 3.1** ⭐ Сколько в таблице `df_raw` полностью одинаковых строк? Выведите **все** копии (включая первые вхождения).

**Задача 3.2** ⭐ Проверьте, уникален ли столбец `order_id`. Сколько номеров заказов повторяется?

**Задача 3.3** ⭐⭐ Заказ `1009` встречается дважды, но строки не полностью одинаковые. Чем они отличаются? Удалите дубликаты по `order_id` так, чтобы осталась **последняя** (самая свежая) запись. Перенумеруйте индекс. Сколько строк осталось?

**Задача 3.4** ⭐⭐⭐ Клиенты `'  Алишер Каримов '` и `'алишер каримов'` — один и тот же человек. Сколько **уникальных** клиентов в столбце `mijoz` до и после очистки имён (`strip` + `title`)? Выведите список клиентов, которые сделали больше одного заказа (после удаления дубликатов заказов из задачи 3.3).

In [ ]:
# Задача 3.1
df = df_raw.copy()
print('Полных дубликатов:', df.duplicated().sum())
display(df[df.duplicated(keep=False)])

# Задача 3.2
print('order_id уникален?', df['order_id'].is_unique)
print('Повторяющихся номеров:', df['order_id'].duplicated().sum())

# Задача 3.3
display(df[df['order_id'] == 1009])
# Отличается только yetkazish_sana: в первой записи дата доставки пустая,
# во второй (обновлённой) — 2026-09-07.
df = df.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
print('Строк осталось:', len(df))

# Задача 3.4
print('Уникальных клиентов до очистки: ', df['mijoz'].nunique())
df['mijoz'] = df['mijoz'].str.strip().str.title()
print('Уникальных клиентов после очистки:', df['mijoz'].nunique())
counts = df['mijoz'].value_counts()
print(counts[counts > 1])

## 4. Исправление типов данных

**Задача 4.1** ⭐ Выведите типы всех столбцов `df_raw`. Какие столбцы имеют неправильный тип и какой тип должен быть у них?

**Задача 4.2** ⭐ Попробуйте выполнить `df_raw['narx'].astype(int)`. Какая ошибка возникла и почему? (Оберните код в `try / except`, чтобы ноутбук не остановился.)

**Задача 4.3** ⭐⭐ Преобразуйте `narx` в число с помощью `pd.to_numeric(..., errors='coerce')` **без предварительной очистки**. Выведите исходные значения, которые превратились в `NaN`. Какие из них были на самом деле корректными ценами?

**Задача 4.4** ⭐⭐ Теперь сначала очистите `narx`: удалите все символы, кроме цифр (подсказка: `.str.replace(r'\D', '', regex=True)`), затем преобразуйте в число. Сколько значений теперь стало `NaN`?

**Задача 4.5** ⭐⭐ Преобразуйте `buyurtma_sana` и `yetkazish_sana` в даты, **явно указав формат** `'%Y-%m-%d'`. Какие значения не удалось распознать? Выведите минимальную и максимальную дату заказа.

**Задача 4.6** ⭐⭐ Столбец `soni` имеет тип `float64`, хотя количество — целое число. Преобразуйте его в тип `Int64`, сохранив пропуски.

**Задача 4.7** ⭐⭐ Преобразуйте `kategoriya` в тип `category`. Сравните объём памяти до и после для ряда `pd.Series(df_raw['kategoriya'].tolist() * 10000)` (используйте `memory_usage(deep=True)`).

**Задача 4.8** ⭐ Дан ряд ответов на вопрос «Нужна ли доставка?»:
```python
otvet = pd.Series(['ha', "Yo'q", 'HA', 'yes', 'no', 'Ha '])
```
Преобразуйте его в `True` / `False`.

In [ ]:
# Задача 4.1
df = df_raw.copy()
print(df.dtypes)
# narx -> число (float64/int64), soni -> целое (Int64),
# buyurtma_sana, yetkazish_sana -> datetime64, kategoriya -> category

# Задача 4.2
try:
    df['narx'].astype(int)
except ValueError as e:
    print('\nОшибка:', e)
# astype строгий: '12 000' содержит пробел, а '?' и '' вообще не числа.

# Задача 4.3
converted = pd.to_numeric(df['narx'], errors='coerce')
lost = df.loc[converted.isna(), 'narx']
print('\nПревратились в NaN:', lost.tolist())
# Корректными ценами были '12 000', '520 000 сум' и '6 500' — мы их потеряли!

# Задача 4.4
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
print('NaN после очистки:', df['narx'].isna().sum())   # только '?' и ''
print(df['narx'].tolist())

# Задача 4.5
for col in ['buyurtma_sana', 'yetkazish_sana']:
    parsed = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
    print(col, '- не распознано:', df.loc[parsed.isna() & df[col].notna(), col].tolist())
    df[col] = parsed
print('Даты заказов:', df['buyurtma_sana'].min().date(), '—', df['buyurtma_sana'].max().date())

# Задача 4.6
df['soni'] = df['soni'].astype('Int64')
print(df['soni'].tolist(), df['soni'].dtype)

# Задача 4.7
big = pd.Series(df_raw['kategoriya'].tolist() * 10000)
print('str:     ', big.memory_usage(deep=True), 'байт')
print('category:', big.astype('category').memory_usage(deep=True), 'байт')
df['kategoriya'] = df['kategoriya'].astype('category')

# Задача 4.8
otvet = pd.Series(['ha', "Yo'q", 'HA', 'yes', 'no', 'Ha '])
print(otvet.str.strip().str.lower().map({'ha': True, 'yes': True, "yo'q": False, 'no': False}))

## 5. Выбросы (outliers)

Для заданий этого раздела используйте очищенную цену:
```python
narx = pd.to_numeric(df_raw['narx'].str.replace(r'\D', '', regex=True), errors='coerce').dropna()
```

**Задача 5.1** ⭐ Выведите `describe()` для `narx`. Сравните среднее и медиану. Что подсказывает разница между `75%` и `max`?

**Задача 5.2** ⭐⭐ Напишите функцию `iqr_bounds(s, k=1.5)`, которая возвращает нижнюю и верхнюю границы по методу IQR. Найдите выбросы в `narx`.

**Задача 5.3** ⭐⭐ Найдите выбросы в `narx` по z-оценке (`|z| > 3`) и по робастной z-оценке (через медиану и MAD, порог 3.5). Выведите максимальное значение `|z|` для обоих методов. Насколько «уверенно» каждый метод нашёл выброс? Объясните разницу.

**Задача 5.4** ⭐⭐⭐ Цены сильно зависят от категории: 1 500 000 — нормально для электроники, но не для продуктов. Найдите выбросы по IQR **внутри каждой категории** (подсказка: `groupby('kategoriya')['narx'].transform(lambda s: s.quantile(0.25))`). Какой заказ оказался выбросом?

**Задача 5.5** ⭐⭐ Даны баллы за экзамен (максимум — 100):
```python
ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])
```
Найдите невозможные значения по **бизнес-правилу** (балл от 0 до 100). Затем ограничьте значения диапазоном `[0, 100]` с помощью `clip`. Подумайте: является ли балл `0` ошибкой?

In [ ]:
narx = pd.to_numeric(df_raw['narx'].str.replace(r'\D', '', regex=True), errors='coerce').dropna()

# Задача 5.1
print(narx.describe())
print('Среднее:', narx.mean(), '| медиана:', narx.median())
# Среднее намного больше медианы, а max в разы больше 75% — признак выброса сверху.

# Задача 5.2
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

lo, hi = iqr_bounds(narx)
print(f'\nГраницы IQR: [{lo:,.0f}; {hi:,.0f}]')
print('Выбросы IQR:', narx[(narx < lo) | (narx > hi)].tolist())

# Задача 5.3
z = (narx - narx.mean()) / narx.std()
print('Выбросы z-score (|z|>3):', narx[z.abs() > 3].tolist(), '| max |z| =', z.abs().max().round(2))
med = narx.median()
mad = (narx - med).abs().median()
robust_z = 0.6745 * (narx - med) / mad
print('Выбросы робастный z:', narx[robust_z.abs() > 3.5].tolist(), '| max |z| =', robust_z.abs().max().round(2))
# Все три метода нашли 1 500 000, но обычный z-score — «еле-еле»: |z| = 3.02,
# чуть выше порога 3. Сам выброс раздул среднее и стандартное отклонение;
# будь выборка чуть меньше, z-score его бы пропустил. Робастный z и IQR надёжнее.

# Задача 5.4
d = df_raw.copy()
d['narx'] = pd.to_numeric(d['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
g = d.groupby('kategoriya')['narx']
q1 = g.transform(lambda s: s.quantile(0.25))
q3 = g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
out = (d['narx'] < q1 - 1.5 * iqr) | (d['narx'] > q3 + 1.5 * iqr)
display(d.loc[out, ['order_id', 'kategoriya', 'narx']])
# Выброс — заказ 1008: продукт за 1 500 000 (вероятно, лишние нули).

# Задача 5.5
ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])
print('Невозможные значения:', ball[~ball.between(0, 100)].tolist())
print('После clip:', ball.clip(0, 100).tolist())
# 0 — допустимое значение (студент мог не решить ни одной задачи или не прийти),
# поэтому формально это не ошибка. Но стоит уточнить, не означает ли 0 «не сдавал».

## 6. Очистка строк: strip, replace, regex

**Задача 6.1** ⭐ Приведите столбец `mijoz` к единому виду: уберите лишние пробелы по краям и внутри строки, каждое слово — с заглавной буквы.

**Задача 6.2** ⭐⭐ Нормализуйте столбец `shahar`. В результате должны остаться только значения `Ташкент`, `Самарканд`, `Бухара`, `Наманган` и `NaN`. Выведите `value_counts(dropna=False)` до и после.
Подсказки: `strip` → `lower` → убрать префикс `'г. '` с помощью regex → словарь замен → `title`.

**Задача 6.3** ⭐⭐ Нормализуйте телефоны к формату `+998XXXXXXXXX`:
1. удалите все символы, кроме цифр;
2. если цифр 12 и номер начинается с `998` — оставьте последние 9 цифр; если цифр 9 — оставьте как есть; иначе — `NaN`;
3. добавьте `'+998'` в начало.

Какие номера оказались некорректными?

**Задача 6.4** ⭐⭐ Дан ряд цен в произвольном формате:
```python
price_txt = pd.Series(['12 000 сум', 'цена: 9500', '7,500', '15000 so\'m', 'договорная', '1 200 000'])
```
Извлеките число из каждой строки с помощью `.str.extract`, очистите разделители и преобразуйте в число.

**Задача 6.5** ⭐⭐ Дан ряд email-адресов:
```python
emails = pd.Series(['ali@mail.uz', ' Bobur@Gmail.com', 'kamola.mail.ru', 'test@', 'sardor_n@tiu.uz '])
```
Очистите адреса (пробелы, нижний регистр) и проверьте, соответствует ли каждый адрес формату `имя@домен.зона` с помощью `.str.match`.

**Задача 6.6** ⭐ Дан ряд адресов:
```python
manzil = pd.Series(['Ташкент, Чиланзар, 12', 'Самарканд, Регистан, 5', 'Бухара, Центр, 7'])
```
Разделите его на три столбца: `shahar`, `tuman`, `uy` с помощью `.str.split(..., expand=True)`.

**Задача 6.7** ⭐⭐⭐ Найдите в `df_raw` всех клиентов, в имени которых есть буквосочетание «ов» (без учёта регистра), и всех клиентов, чьё **имя** (первое слово) начинается на «А» или «Б».

In [ ]:
df = df_raw.copy()

# Задача 6.1
df['mijoz'] = (df['mijoz'].str.strip()
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.title())
print(df['mijoz'].tolist())

# Задача 6.2
print(df['shahar'].value_counts(dropna=False))
df['shahar'] = (df['shahar'].str.strip()
                            .str.lower()
                            .str.replace(r'^г\.\s*', '', regex=True)
                            .replace({'tashkent': 'ташкент', 'n/a': np.nan, '-': np.nan})
                            .str.title())
print(df['shahar'].value_counts(dropna=False))

# Задача 6.3
digits = df['telefon'].str.replace(r'\D', '', regex=True)
local = np.where((digits.str.len() == 12) & digits.str.startswith('998'), digits.str[-9:],
         np.where(digits.str.len() == 9, digits, None))
df['telefon_clean'] = '+998' + pd.Series(local, index=df.index)
display(df[['telefon', 'telefon_clean']])
print('Некорректные:', df.loc[df['telefon_clean'].isna(), 'telefon'].tolist())

# Задача 6.4
price_txt = pd.Series(['12 000 сум', 'цена: 9500', '7,500', "15000 so'm", 'договорная', '1 200 000'])
price_num = (price_txt.str.extract(r'(\d[\d\s,]*)')[0]
                      .str.replace(r'[\s,]', '', regex=True)
                      .pipe(pd.to_numeric, errors='coerce'))
print(price_num.tolist())

# Задача 6.5
emails = pd.Series(['ali@mail.uz', ' Bobur@Gmail.com', 'kamola.mail.ru', 'test@', 'sardor_n@tiu.uz '])
clean_emails = emails.str.strip().str.lower()
print(pd.DataFrame({'email': clean_emails,
                    'ok': clean_emails.str.match(r'^[\w.+-]+@[\w-]+\.[\w.]+$')}))

# Задача 6.6
manzil = pd.Series(['Ташкент, Чиланзар, 12', 'Самарканд, Регистан, 5', 'Бухара, Центр, 7'])
parts = manzil.str.split(', ', expand=True)
parts.columns = ['shahar', 'tuman', 'uy']
print(parts)

# Задача 6.7
names = df['mijoz'].dropna()
print(names[names.str.contains('ов', case=False)].unique())
print(names[names.str.match(r'^[АБ]')].unique())

## 7. Проверка согласованности данных

Для заданий этого раздела подготовьте таблицу:
```python
df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
for col in ['buyurtma_sana', 'yetkazish_sana']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
```

**Задача 7.1** ⭐ Найдите заказы с неположительным количеством (`soni <= 0`).

**Задача 7.2** ⭐ Найдите заказы, у которых дата доставки **раньше** даты заказа.

**Задача 7.3** ⭐⭐ Проверьте правило `summa == narx × soni`. Выведите строки, где правило нарушено (строки с пропусками в этих столбцах не учитывайте).

**Задача 7.4** ⭐⭐ Допустимые категории: `Продукты`, `Одежда`, `Электроника`, `Бытовая техника`. Проверьте, что все категории допустимы.

**Задача 7.5** ⭐⭐⭐ Напишите функцию `check(df)`, которая возвращает `pd.Series` с количеством проблем по каждой проверке:
* повторяющиеся `order_id`;
* `soni <= 0`;
* `narx <= 0`;
* `summa` не равна `narx × soni`;
* доставка раньше заказа;
* дата заказа в будущем;
* общее количество пропусков.

Запустите её на `df`.

**Задача 7.6** ⭐⭐ Добавьте проверки `assert`: номера заказов уникальны, количество положительно. Убедитесь, что на неочищенных данных `assert` срабатывает (оберните в `try / except AssertionError`).

In [ ]:
df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
for col in ['buyurtma_sana', 'yetkazish_sana']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

# Задача 7.1
display(df[df['soni'] <= 0])

# Задача 7.2
display(df[df['yetkazish_sana'] < df['buyurtma_sana']])

# Задача 7.3
full = df[['narx', 'soni', 'summa']].notna().all(axis=1)
bad_sum = full & ((df['narx'] * df['soni'] - df['summa']).abs() > 0.01)
display(df.loc[bad_sum, ['order_id', 'narx', 'soni', 'summa']])

# Задача 7.4
allowed = ['Продукты', 'Одежда', 'Электроника', 'Бытовая техника']
print('Все категории допустимы:', df['kategoriya'].isin(allowed).all())

# Задача 7.5
def check(df):
    return pd.Series({
        'дубликаты order_id':     df['order_id'].duplicated().sum(),
        'soni <= 0':              (df['soni'] <= 0).sum(),
        'narx <= 0':              (df['narx'] <= 0).sum(),
        'summa != narx*soni':     ((df['narx'] * df['soni'] - df['summa']).abs() > 0.01).sum(),
        'доставка раньше заказа': (df['yetkazish_sana'] < df['buyurtma_sana']).sum(),
        'заказ в будущем':        (df['buyurtma_sana'] > pd.Timestamp('today')).sum(),
        'пропуски (всего)':       df.isna().sum().sum(),
    })

print(check(df))

# Задача 7.6
try:
    assert df_raw['order_id'].is_unique, 'Есть повторяющиеся order_id'
except AssertionError as e:
    print('\nСработал assert:', e)
try:
    assert (df['soni'].dropna() > 0).all(), 'Есть неположительное количество'
except AssertionError as e:
    print('Сработал assert:', e)

## 8. Итоговое задание: конвейер очистки

**Задача 8.1** ⭐⭐⭐ Напишите функцию `clean(raw)`, которая возвращает очищенную копию `df_raw`. Соблюдайте порядок шагов из лекции:

1. **Скрытые пропуски** → `NaN`.
2. **Строки**: `mijoz` (пробелы, регистр), `shahar` (единое написание), `telefon` (формат `+998XXXXXXXXX`).
3. **Типы**: `narx` → число, даты → `datetime`, `kategoriya` → `category`.
4. **Дубликаты**: по `order_id`, оставить последнюю запись.
5. **Ошибки и выбросы**:
   * `soni <= 0` → `NaN`;
   * выбросы цены по IQR **внутри категории** → `NaN`;
   * дата доставки раньше даты заказа → `NaN` в `yetkazish_sana`.
6. **Пропуски**:
   * `narx` — медиана категории;
   * `soni` — медиана, затем тип `int`;
   * `mijoz`, `shahar` — `'Неизвестно'`;
   * `buyurtma_sana` — предыдущая дата (`ffill`); `yetkazish_sana` оставить пустой (заказ мог быть ещё не доставлен).
7. **Пересчитать** `summa = narx × soni`.

**Задача 8.2** ⭐⭐ Запустите `check()` из задачи 7.5 **до** и **после** очистки и выведите результаты рядом.

**Задача 8.3** ⭐⭐ Добавьте в конце `assert`-проверки: `order_id` уникален, `soni > 0`, в `narx` и `summa` нет пропусков.

**Задача 8.4** ⭐⭐ Составьте таблицу «было — стало»: количество строк, пропусков, уникальных городов, уникальных клиентов, средняя и медианная сумма заказа.

**Задача 8.5** ⭐⭐⭐ (Для размышления) Заказ `1008`: продукт за 1 500 000. Мы заменили цену медианой категории. Предложите другой способ обработки и объясните, когда он был бы лучше.

In [ ]:
def clean(raw):
    df = raw.copy()

    # 1. Скрытые пропуски
    df = df.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})

    # 2. Строки
    df['mijoz'] = df['mijoz'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
    df['shahar'] = (df['shahar'].str.strip().str.lower()
                                .str.replace(r'^г\.\s*', '', regex=True)
                                .replace({'tashkent': 'ташкент'})
                                .str.title())
    digits = df['telefon'].str.replace(r'\D', '', regex=True)
    ok12 = (digits.str.len() == 12) & digits.str.startswith('998')
    ok9 = digits.str.len() == 9
    df['telefon'] = ('+998' + digits.str[-9:]).where(ok12 | ok9)

    # 3. Типы
    df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
    for col in ['buyurtma_sana', 'yetkazish_sana']:
        df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
    df['kategoriya'] = df['kategoriya'].astype('category')

    # 4. Дубликаты
    df = df.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)

    # 5. Ошибки и выбросы -> NaN
    df.loc[df['soni'] <= 0, 'soni'] = np.nan
    g = df.groupby('kategoriya', observed=True)['narx']
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    df['narx'] = df['narx'].where(df['narx'].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr) | df['narx'].isna())
    df.loc[df['yetkazish_sana'] < df['buyurtma_sana'], 'yetkazish_sana'] = pd.NaT

    # 6. Пропуски
    df['narx'] = df['narx'].fillna(df.groupby('kategoriya', observed=True)['narx'].transform('median'))
    df['soni'] = df['soni'].fillna(df['soni'].median()).astype(int)
    df['mijoz'] = df['mijoz'].fillna('Неизвестно')
    df['shahar'] = df['shahar'].fillna('Неизвестно')
    df['buyurtma_sana'] = df['buyurtma_sana'].ffill()

    # 7. Пересчёт суммы
    df['summa'] = df['narx'] * df['soni']
    return df


# Подготовка «сырых» данных для check(): только типы, без исправлений
def prepare_for_check(raw):
    d = raw.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
    d['narx'] = pd.to_numeric(d['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
    for col in ['buyurtma_sana', 'yetkazish_sana']:
        d[col] = pd.to_datetime(d[col], format='%Y-%m-%d', errors='coerce')
    return d

clean_df = clean(df_raw)
display(clean_df)
print(clean_df.dtypes)

# Задача 8.2
print(pd.DataFrame({'до': check(prepare_for_check(df_raw)), 'после': check(clean_df)}))
# Оставшиеся пропуски после очистки — это telefon (некорректные номера)
# и yetkazish_sana (заказ не доставлен или дата была ошибочной). Это осознанное решение.

# Задача 8.3
assert clean_df['order_id'].is_unique
assert (clean_df['soni'] > 0).all()
assert clean_df['narx'].notna().all() and clean_df['summa'].notna().all()
print('\nВсе проверки assert пройдены')

# Задача 8.4
raw_c = prepare_for_check(df_raw)
summary = pd.DataFrame({
    'было': [len(df_raw), raw_c.isna().sum().sum(), df_raw['shahar'].nunique(),
             df_raw['mijoz'].nunique(), raw_c['summa'].mean(), raw_c['summa'].median()],
    'стало': [len(clean_df), clean_df.isna().sum().sum(), clean_df['shahar'].nunique(),
              clean_df['mijoz'].replace('Неизвестно', np.nan).nunique(), clean_df['summa'].mean(), clean_df['summa'].median()],
}, index=['строк', 'пропусков', 'уникальных городов', 'уникальных клиентов',
          'средняя сумма', 'медианная сумма'])
print(summary.round(0))

**Ответ к задаче 8.5 (пример):**
Цена 1 500 000 за продукт похожа на опечатку с лишними нулями (вероятно, 15 000). Если в данных есть **история цен этого товара** или **справочник цен**, лучше исправить значение по справочнику, а не заменять медианой. Если таких источников нет, можно не заменять цену, а **отметить заказ флагом** `narx_vybros = True` и исключить его из расчёта средних, сохранив в данных. Главное — задокументировать решение и уточнить у владельца данных.

## 9. Вопросы для самопроверки (ответы)

1. `NaN` означает «неизвестное значение», а два неизвестных значения нельзя считать равными. Проверять нужно через `isna()` / `notna()`.
2. Значения, которые означают «нет данных», но записаны как обычный текст или число: `'N/A'`, `'-'`, `'?'`, `''`, `0`, `-1`, `999`, `'1900-01-01'`.
3. MCAR — пропуск полностью случаен (сбой датчика); MAR — зависит от других столбцов (молодые реже указывают доход); MNAR — зависит от самого значения (люди с высоким доходом скрывают его).
4. Пропуски находятся в **разных** строках; `dropna()` удаляет строку, если пусто хоть одно значение, поэтому теряется объединение всех строк с пропусками.
5. Когда в данных есть выбросы или распределение «перекошено» (цены, доходы, зарплаты): медиана устойчива к экстремальным значениям.
6. Из-за механизма Copy-on-Write `df['col']` — отдельный объект; изменение «на месте» не попадает в таблицу. Нужно писать `df['col'] = df['col'].fillna(0)`.
7. `duplicated()` сравнивает строки посимвольно: `'Алишер'` и `' алишер '` для него разные. После очистки неявные дубликаты становятся явными.
8. `coerce` молча превращает в `NaN` и корректные, но «грязные» значения (например, `'4 500'`). Нужно сначала очистить строку, а после преобразования проверить, какие значения стали `NaN`.
9. Выброс сам увеличивает среднее и стандартное отклонение, поэтому его z-оценка оказывается небольшой; особенно на маленьких выборках. Помогают IQR или робастная z-оценка.
10. `Series.str.replace()` заменяет **подстроку** внутри каждой строки; `Series.replace()` заменяет **значение ячейки целиком** при точном совпадении.
11. `summa == narx × soni`; `yetkazish_sana >= buyurtma_sana`.
12. Чтобы скрипт сразу остановился, если в новых данных появится проблема, которую очистка не исправила, — а не после того, как неверный отчёт будет отправлен.